# Owner-aware temporal monitoring of personal food/drink containers using edge AI to identify suspicious interactions and potential unauthorized access.

## Project Overview

An intelligent surveillance and security system designed to detect unauthorized tampering with food items (cups, bottles, lunch boxes). The system processes data in the following sequential architecture:

1. **Identify food/drink containers** using object detection and tracking
2. **Identify the authenticated owner** via facial authentication and tracking
3. **Detect hand gestures** and track hand movements
4. **Calculate Personal Space** around the detected food items
5. **Alert on tampering** when unauthenticated individuals breach the personal space or touch items
6. **Log events and send alerts** via Telegram

### Key Features:
- Sequential processing architecture optimized for context awareness
- Food item detection and tracking (YOLOv8s Custom ONNX model + Tracking)
- Owner authentication via face recognition (InsightFace)
- Hand pose tracking (MediaPipe)
- Personal Space bounding box calculation to prevent proximity tampering
- Tampering event logging and Telegram notifications
- Event buffering and evidence capture

---

## Installation & Setup

### Install Dependencies

First, install all required packages from requirements.txt:

In [1]:
# !pip install -r requirements.txt

# Core packages needed:
# - opencv-python>=4.8.0
# - mediapipe>=0.10.0
# - insightface>=0.7.3
# - onnxruntime>=1.15.0
# - psutil>=5.9.0
# - gputil (optional, for GPU monitoring)
# - python-telegram-bot (for alerts)
# - matplotlib & seaborn (for visualization)

print("Dependencies installation guide:")
print("Run: pip install -r requirements.txt")

Dependencies installation guide:
Run: pip install -r requirements.txt


### Download Pre-trained Models

The system uses pre-trained models from MediaPipe and a custom YOLOv8 model:

In [2]:
import os
import urllib.request

def download_models():
    """Downloads AI models directly to the root folder."""
    models = {
        "hand_landmarker.task": "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        "face_landmarker.task": "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task",
        "efficientdet.tflite": "https://storage.googleapis.com/mediapipe-tasks/object_detector/efficientdet_lite0_uint8.tflite"
    }
    for filename, url in models.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}... Please wait.")
            urllib.request.urlretrieve(url, filename)
        else:
            print(f"[OK] {filename} already exists.")

# Uncomment to download models
# download_models()
print("Models are pre-trained and already available in the project directory.")

Models are pre-trained and already available in the project directory.


---

## Module 1: Hand Tracking (MediaPipe)

### Purpose:
Detects and tracks hand landmarks in real-time using MediaPipe's HandLandmarker model. Supports up to 6 hands simultaneously.

In [3]:
import cv2
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

class HandTracker:
    """Detects and tracks hand landmarks using MediaPipe."""
    
    def __init__(self):
        options = vision.HandLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path='hand_landmarker.task'),
            running_mode=vision.RunningMode.VIDEO,
            num_hands=6  # Support up to 6 hands
        )
        self.detector = vision.HandLandmarker.create_from_options(options)
        
        # Hand skeleton connections (21 landmarks)
        self.connections = [
            (0, 1), (1, 2), (2, 3), (3, 4),       # Thumb
            (0, 5), (5, 6), (6, 7), (7, 8),       # Index finger
            (5, 9), (9, 10), (10, 11), (11, 12),  # Middle finger
            (9, 13), (13, 14), (14, 15), (15, 16),# Ring finger
            (13, 17), (0, 17), (17, 18), (18, 19), (19, 20)  # Pinky + palm
        ]

    def process_and_return(self, mp_image, timestamp_ms):
        """Process image and return hand landmarks in pixel coordinates."""
        h, w = mp_image.height, mp_image.width
        results = self.detector.detect_for_video(mp_image, timestamp_ms)
        
        hands_data = []
        if results.hand_landmarks:
            for hand_lms in results.hand_landmarks:
                # Convert normalized coordinates to pixel coordinates
                pixel_lms = [(int(lm.x * w), int(lm.y * h)) for lm in hand_lms]
                hands_data.append(pixel_lms)
        return hands_data

    def close(self):
        """Clean up resources."""
        self.detector.close()

print("HandTracker class loaded successfully!")

HandTracker class loaded successfully!


---

## Module 2: Face Authentication and Tracking (InsightFace + MediaPipe)

### Purpose:
Authenticates the owner and tracks faces in the scene to verify if the individual approaching the food items is authorized.

In [4]:
import os
import cv2
import numpy as np
import threading
import time
from insightface.app import FaceAnalysis
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

class FaceTracker:
    """Dual-model face authentication and tracking system."""
    
    def __init__(self, user_image_path="user.jpg"):
        # 1. Initialize MediaPipe (Ultra-fast foreground tracking)
        options = vision.FaceLandmarkerOptions(
            base_options=python.BaseOptions(model_asset_path='face_landmarker.task'),
            running_mode=vision.RunningMode.VIDEO,
            num_faces=1
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)
        
        # 2. Initialize InsightFace
        print("\nInitializing InsightFace (buffalo_l model)...")
        self.app = FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
        self.app.prepare(ctx_id=0, det_size=(320, 320))
        
        # 3. State Management Variables
        self.user_embedding = None
        self.current_label = "Scanning..."
        self.current_color = (0, 255, 255)  # Yellow
        self.is_authenticated = False
        
        # Threading controls
        self.recognition_thread = None
        self.is_processing_identity = False
        
        # Security Cooldown controls
        self.last_auth_time = 0
        self.auth_cooldown = 3.0  # Seconds to wait before re-verifying

        # Load Master Identity
        if os.path.exists(user_image_path):
            print(f"Loading identity from {user_image_path}...")
            user_img = cv2.imread(user_image_path)
            faces = self.app.get(user_img)
            if faces:
                self.user_embedding = faces[0].embedding
                print(f"[SUCCESS] Identity securely loaded into memory!")
            else:
                self.current_label = "BAD user.jpg"
                self.current_color = (150, 150, 150)
        else:
            print(f"[CRITICAL WARNING] '{user_image_path}' not found!")
            self.current_label = "NO user.jpg FOUND"
            self.current_color = (150, 150, 150)

    def _recognize_face(self, face_crop):
        """BACKGROUND TASK: Runs heavy math without freezing the webcam."""
        self.is_processing_identity = True
        try:
            detected_faces = self.app.get(face_crop)
                
            if detected_faces:
                current_embedding = detected_faces[0].embedding
                sim = np.dot(self.user_embedding, current_embedding) / (
                    np.linalg.norm(self.user_embedding) * np.linalg.norm(current_embedding)
                )
                
                if sim > 0.40:
                    self.current_label = f"Owner ({sim:.2f})"
                    self.current_color = (0, 255, 0)  # Green
                    self.is_authenticated = True
                else:
                    self.current_label = f"INTRUDER! ({sim:.2f})"
                    self.current_color = (0, 0, 255)  # Red
                    self.is_authenticated = False
            else:
                self.current_label = "Scan Failed. Retrying..."
                self.current_color = (0, 165, 255)  # Orange
                self.is_authenticated = False
        except Exception as e:
            print(f"Error in face recognition thread: {e}")
            self.current_label = "Scan Error"
            self.current_color = (0, 165, 255)
        finally:
            self.is_processing_identity = False

    def process_and_draw(self, img, clean_img, mp_image, timestamp_ms, frame_count):
        """Process frame and update authentication status."""
        h, w, _ = img.shape
        results = self.detector.detect_for_video(mp_image, timestamp_ms)
        face_detected_this_frame = False
        current_time = time.time()

        if results.face_landmarks:
            for face_lms in results.face_landmarks:
                face_detected_this_frame = True
                
                x_coords = [lm.x for lm in face_lms]
                y_coords = [lm.y for lm in face_lms]
                left = int((min(x_coords) - 0.1) * w)
                right = int((max(x_coords) + 0.1) * w)
                top = int((min(y_coords) - 0.25) * h)
                bottom = int((max(y_coords) + 0.1) * h)
                left, right = max(0, left), min(w - 1, right)
                top, bottom = max(0, top), min(h - 1, bottom)

                if self.user_embedding is not None and not self.is_authenticated and not self.is_processing_identity:
                    if current_time - self.last_auth_time > self.auth_cooldown:
                        face_crop = clean_img[top:bottom, left:right]
                        if face_crop.size != 0:
                            self.current_label = "Authenticating..."
                            self.current_color = (0, 255, 255)
                            self.last_auth_time = current_time
                            self.recognition_thread = threading.Thread(target=self._recognize_face, args=(face_crop,))
                            self.recognition_thread.start()
                
                cv2.rectangle(img, (left, top), (right, bottom), self.current_color, 2)
                cv2.putText(img, self.current_label, (left, top - 15), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.8, self.current_color, 2, cv2.LINE_AA)

        if not face_detected_this_frame:
            self.is_authenticated = False
            self.last_auth_time = 0
            if not self.is_processing_identity:
                self.current_label = "Scanning..."
                self.current_color = (0, 255, 255)

    def close(self):
        """Clean up resources."""
        self.detector.close()
        if self.recognition_thread is not None and self.recognition_thread.is_alive():
            self.recognition_thread.join(timeout=1.0)

print("FaceTracker class loaded successfully!")

FaceTracker class loaded successfully!


---

## Module 3: Food Object Detection (YOLOv8s ONNX)

### Purpose:
Acts as the primary initiator of the security pipeline by identifying food containers first.

In [5]:
import cv2
import numpy as np
import onnxruntime as ort

class FoodDetector:
    """YOLOv8 ONNX model for food container detection."""
    
    def __init__(self, model_path="best.onnx"):
        print("Initializing YOLOv8s Custom (ONNX)...")
        self.session = ort.InferenceSession(model_path, providers=['CPUExecutionProvider'])
        self.input_name = self.session.get_inputs()[0].name
        
        # Class mapping
        self.target_classes = {
            0: "0",
            1: "Bottle", 
            2: "Lunch Box",
            3: "cup"
        }
        self.valid_classes = ["Bottle", "Lunch Box", "cup"]

    def process_and_draw(self, img, clean_img):
        """Detect food containers in image."""
        img_h, img_w = clean_img.shape[:2]
        
        # Prepare input
        rgb_img = cv2.cvtColor(clean_img, cv2.COLOR_BGR2RGB)
        input_img = cv2.resize(rgb_img, (640, 640))
        input_img = input_img.astype(np.float32) / 255.0
        input_img = input_img.transpose(2, 0, 1)  # HWC -> CHW
        input_tensor = np.expand_dims(input_img, axis=0)

        # Run inference
        outputs = self.session.run(None, {self.input_name: input_tensor})[0]
        predictions = np.squeeze(outputs).T
        
        # Parse output
        boxes = predictions[:, :4]  # cx, cy, w, h
        scores = predictions[:, 4:]
        class_ids = np.argmax(scores, axis=1)
        confidences = np.max(scores, axis=1)
        
        mask = (confidences > 0.60) & np.isin(class_ids, list(self.target_classes.keys()))
        filtered_boxes = boxes[mask]
        filtered_conf = confidences[mask]
        filtered_class_ids = class_ids[mask]
        
        x_factor = img_w / 640.0
        y_factor = img_h / 640.0
        
        nms_boxes = []
        nms_boxes_offset = []
        
        for i, row in enumerate(filtered_boxes):
            cx, cy, w, h = row
            left = int((cx - w / 2) * x_factor)
            top = int((cy - h / 2) * y_factor)
            width = int(w * x_factor)
            height = int(h * y_factor)
            
            nms_boxes.append([left, top, width, height])
            offset = int(filtered_class_ids[i] * 4096)
            nms_boxes_offset.append([left + offset, top + offset, width, height])
        
        indices = cv2.dnn.NMSBoxes(nms_boxes_offset, filtered_conf.tolist(), 0.35, 0.45)
        
        detected_items = []
        
        if len(indices) > 0:
            for i in indices.flatten():
                box = nms_boxes[i]
                cls_id = filtered_class_ids[i]
                x, y, w, h = box[0], box[1], box[2], box[3]
                
                category = self.target_classes.get(cls_id, "unknown")
                if category in self.valid_classes:
                    detected_items.append({"box": (x, y, w, h), "category": category})
                
        return detected_items

    def close(self):
        pass

print("FoodDetector class loaded successfully!")

FoodDetector class loaded successfully!


---

## Module 4: Performance Metrics & Logging

### Purpose:
Monitors and logs system performance metrics.

In [6]:
import csv
import os
import platform
import subprocess
import time
from datetime import datetime
import cv2
import psutil

try:
    import GPUtil
    GPU_AVAILABLE = True
except ImportError:
    GPU_AVAILABLE = False

class PerformanceEvaluator:
    """Tracks performance metrics and logs events."""
    
    def __init__(self, log_dir="data-logs", spec_file="hardware_specs.txt", 
                 log_file="hardware_logs.csv", tamper_file="tamper_events.csv"):
        self.log_dir = log_dir
        self.spec_path = os.path.join(log_dir, spec_file)
        self.log_path = os.path.join(log_dir, log_file)
        self.tamper_path = os.path.join(log_dir, tamper_file)

        os.makedirs(self.log_dir, exist_ok=True)

        self.fps_list = []
        self.fps = 0.0
        self.max_ram_used_gb = 0.0
        self.gpu_name = "N/A"
        self.max_gpu_load = 0.0
        self.max_gpu_mem = 0.0

        self._setup_tamper_csv()
        self._generate_specs_file()

    def _get_processor_name(self):
        try:
            if platform.system() == "Windows":
                return platform.processor()
            elif platform.system() == "Darwin":
                command = "sysctl -n machdep.cpu.brand_string"
                return subprocess.check_output(command, shell=True).decode().strip()
            elif platform.system() == "Linux":
                command = "cat /proc/cpuinfo | grep 'model name' | uniq"
                return subprocess.check_output(command, shell=True).decode().split(':')[1].strip()
        except Exception:
            return platform.machine()
        return "Unknown Processor"

    def _generate_specs_file(self):
        device_name = platform.node()
        os_platform = f"{platform.system()} {platform.release()}"
        processor = self._get_processor_name()
        cpu_cores = psutil.cpu_count(logical=True)
        total_memory_gb = round(psutil.virtual_memory().total / (1024 ** 3), 2)

        gpu_format = "N/A"
        if GPU_AVAILABLE:
            try:
                gpus = GPUtil.getGPUs()
                if gpus:
                    gpu_format = gpus[0].name
                    self.gpu_name = gpu_format
            except Exception:
                pass

        with open(self.spec_path, "a", encoding="utf-8") as f:
            f.write(f"\n--- New Session: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ---\n")
            f.write(f"Device Name: {device_name}\n")
            f.write(f"OS Platform: {os_platform}\n")
            f.write(f"Processor: {processor}\n")
            f.write(f"CPU Cores: {cpu_cores}\n")
            f.write(f"Total Memory: {total_memory_gb} GB\n")
            f.write(f"GPU: {gpu_format}\n")

    def _setup_tamper_csv(self):
        if not os.path.isfile(self.tamper_path):
            with open(self.tamper_path, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(["Tamper Timestamp", "Target Object Category", "Tampered or not", "Alert Type"])

    def update_and_draw(self, img, loop_start_time):
        current_time = time.time()
        time_diff = current_time - loop_start_time

        if time_diff > 0:
            current_fps = 1.0 / time_diff
            self.fps = (self.fps * 0.9) + (current_fps * 0.1)
        self.fps_list.append(self.fps)

        cv2.putText(img, f"FPS: {int(self.fps)}", (img.shape[1] - 150, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
        return img

    def log_tamper_event(self, category, tampered_status="YES", alert_type="Direct Touch"):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        with open(self.tamper_path, mode="a", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow([timestamp, category, tampered_status, alert_type])

    def finalize_test_log(self, tampered_status):
        pass

    def close(self):
        pass

print("PerformanceEvaluator class loaded successfully!")

PerformanceEvaluator class loaded successfully!


---

## Main Pipeline: Complete System Integration

### Updated Sequence Architecture:

```
Video Input (Webcam)
    ↓
┌────────────────────────────────────────────────────────┐
│  1. Food Detection (YOLOv8s ONNX) & Object Tracking    │
│  ↓                                                     │
│  2. Face Authentication (InsightFace) & Tracking       │
│  ↓                                                     │
│  3. Hand Detection (MediaPipe)                         │
│  ↓                                                     │
│  4. Personal Space Calculation (50px Margin)           │
│  ↓                                                     │
│  Tampering Logic:                                      │
│    IF hand breaches Personal Space AND user NOT owner  │
│    THEN trigger PROXIMITY alert                        │
│    IF hand touches food box AND user NOT owner         │
│    THEN trigger TAMPER alert                           │
└────────────────────────────────────────────────────────┘
```

In [7]:
import os

TELEGRAM_TOKEN = os.environ.get("TELEGRAM_TOKEN")
TELEGRAM_CHAT_ID = os.environ.get("TELEGRAM_CHAT_ID")

print("Telegram alerts use local environment variables when configured.")

Main pipeline function defined.


---

## Dataset Testing Pipeline
Test detection model on a batch of static images.

In [8]:
import os
import cv2
import time

def run_dataset_test():
    food_detector = FoodDetector()
    print("Dataset testing logic loaded.")
    # Note: Logic identical to prior block for brevity.


---

## Model Training Pipeline
Trains YOLOv8s with Hard-negative mining and partial occlusions.

In [9]:
training_code = """
import os
from roboflow import Roboflow
from ultralytics import YOLO

roboflow_api_key = os.environ.get("ROBOFLOW_API_KEY")
if not roboflow_api_key:
    raise RuntimeError("Set ROBOFLOW_API_KEY before downloading the training dataset.")

rf = Roboflow(api_key=roboflow_api_key)
# Continue with the project/version download and YOLO training configuration.
"""
print("Model training code requires ROBOFLOW_API_KEY from the environment.")

Model Training Code Ready.
